In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Pag-initialize ng Qubit

<Accordion>
<AccordionItem title="Mga bersyon ng package">

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Kapag nag-execute ng circuit sa isang IBM&reg; quantum processing unit (QPU), karaniwang may implicit reset na idiniinsert sa simula ng circuit para matiyak na naininitialize sa zero ang mga qubit. Kinokontrol ito ng `init_qubits` flag, na itinakda bilang isang [primitive execution option](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

Gayunpaman, ang mga depekto sa proseso ng reset ay maaaring magdulot ng mga state-preparation error. Para mabawasan ang error, naglalagay din ang QPU ng repetition delay time (o `rep_delay`) sa pagitan ng mga circuit. Ang bawat backend ay may iba-ibang default na `rep_delay`, pero karaniwang itinakda ito para balansehin ang reset fidelity at kabuuang oras ng execution. Patakbuhin ang `backend.default_rep_delay` para mahanap ang default na `rep_delay` ng isang partikular na QPU.

Dahil lahat ng IBM QPU ay gumagamit ng dynamic repetition rate execution, maaari mong baguhin ang `rep_delay` para sa bawat job. Ang mga circuit na isinusumite mo sa isang primitive job ay pinagsama-sama para sa execution sa QPU. Ang mga circuit na ito ay ine-execute sa pamamagitan ng pag-ulit-ulit sa mga circuit para sa bawat shot na hinihingi; ang execution ay column-wise sa isang matrix ng mga circuit at shot, gaya ng makikita sa sumusunod na larawan.

![Ang unang column ay kumakatawan sa shot 0. Ang mga circuit ay pinapatakbo nang sunud-sunod mula 0 hanggang 3. Ang pangalawang column ay kumakatawan sa shot 1. Ang mga circuit ay pinapatakbo nang sunud-sunod mula 0 hanggang 3. Ang natitirang mga column ay sumusunod sa parehong pattern.](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Column-wise execution matrix")

Dahil ang `rep_delay` ay idiniinsert sa pagitan ng mga circuit, ang bawat shot ng execution ay nararamdaman ang delay na ito. Kaya naman, habang binababa mo ang `rep_delay`, bumababa ang kabuuang oras ng QPU execution, sa gastos ng pagtaas ng rate ng state preparation error, gaya ng inilalarawan ng sumusunod na larawan:

![Ipinapakita ng larawang ito na habang bumababa ang halaga ng `rep_delay`, tumataas ang rate ng state preparation error.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Repetition delay versus error rate")

Kung itakda mo ang parehong `rep_delay=0` at `init_qubits=False`, ang mga circuit ay "pinagsasama," dahil magsisimula ang mga qubit sa huling state mula sa nakaraang shot.

Tandaan na kahit ang mga circuit sa isang primitive job ay pinagsama-sama para sa QPU execution, walang garantiya sa pagkakasunud-sunod ng pag-execute ng mga circuit mula sa mga PUB. Halimbawa, kung isumite mo ang `pubs=[pub1, pub2]`, maaaring hindi mauuna ang mga circuit mula sa `pub1` bago ang mula sa `pub2`. Wala ring garantiya na ang mga circuit mula sa parehong job ay tatakbo bilang isang batch sa QPU.

## Pagtukoy ng `rep_delay` para sa isang primitive job
> **Note:** Palaging i-verify ang sinusuportahang hanay ng `rep_delay` para sa partikular na QPU na ginagamit mo. Hindi pareho ang mga halagang ito para sa bawat QPU at maaari ring magbago sa paglipas ng panahon.
> 
> Tandaan na ang pagtaas ng `rep_delay` ay direktang makakaapekto sa iyong oras ng execution at paggamit ng kapasidad.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Mga susunod na hakbang
> **Tip:** - Subukan ang isang halimbawa sa tutorial na [Quantum approximate optimization algorithm (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Suriin kung paano [magsimula sa Estimator](/guides/get-started-with-estimator).
>   - Suriin kung paano [magsimula sa Sampler](/guides/get-started-with-sampler).